# Repository check

Fast checks for the article-style metro forecasting repository. Set `RUN_FULL_EXPERIMENT = True` only when local data are available and a real forecasting run is needed.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

PROJECT_ROOT

PosixPath('/home/ubuntu/DIPLOM/metro_forecasting_research')

In [2]:
from metroflow.studies.metro_short_history_forecasting.entrypoint import (
    MultiStationForecastConfig,
    run_multistation_forecasting_experiment,
    save_multistation_forecasting_result,
)
from metroflow.studies.metro_short_history_forecasting.evaluation import (
    evaluate_predictions,
    station_bootstrap_ci,
)

cfg = MultiStationForecastConfig()
cfg.random_state

3228

In [3]:
assert cfg.random_state == 3228
assert callable(run_multistation_forecasting_experiment)
assert callable(save_multistation_forecasting_result)

## Compact central-scenario fixture

In [4]:
import pandas as pd

fixture = PROJECT_ROOT / "tests" / "fixtures" / "h060m_b015m"
micro = pd.read_csv(fixture / "micro_metrics.csv").rename(columns={"Unnamed: 0": "model"})
central = micro.loc[micro["model"].eq("Residual XGB ETS LW")].iloc[0]
central[["model", "MAE", "LW-MAE", "HL-MAE"]]

model     Residual XGB ETS LW
MAE                 53.973203
LW-MAE              79.511095
HL-MAE              93.752175
Name: 0, dtype: object

In [5]:
assert round(float(central["LW-MAE"]), 2) == 79.51
assert round(float(central["MAE"]), 2) == 53.97

## Toy metric and bootstrap check

In [6]:
toy = pd.DataFrame({
    "station_key": ["s1", "s1", "s2", "s2", "s3", "s3"],
    "y_true": [100.0, 120.0, 40.0, 60.0, 80.0, 90.0],
    "hl_actual": [0, 1, 0, 1, 0, 1],
    "station_train_mean_target": [100.0, 100.0, 50.0, 50.0, 80.0, 80.0],
    "ETS local": [98.0, 116.0, 44.0, 55.0, 75.0, 95.0],
    "Residual XGB ETS LW": [99.0, 118.0, 42.0, 58.0, 78.0, 91.0],
})

toy_cfg = MultiStationForecastConfig(n_bootstrap=32, run_bootstrap=True)
models = ["ETS local", "Residual XGB ETS LW"]
micro_toy, macro_toy = evaluate_predictions(toy, models, toy_cfg)
assert micro_toy.index[0] == "Residual XGB ETS LW"
assert macro_toy.index[0] == "Residual XGB ETS LW"
micro_toy

,MAE,RMSE,wMAPE,LW-MAE,HL-MAE,HL-underMAE,HL-underRate,HL-asymMAE
Residual XGB ETS LW,1.666667,1.732051,0.020408,1.754167,1.666667,1.333333,0.666667,3.000000
ETS local,4.166667,4.301163,0.051020,4.437500,4.666667,3.000000,0.666667,7.666667


In [7]:
boot_1 = station_bootstrap_ci(toy, models, toy_cfg, baseline_cols=["ETS local"])
boot_2 = station_bootstrap_ci(toy, models, toy_cfg, baseline_cols=["ETS local"])
pd.testing.assert_frame_equal(boot_1["metric_ci"], boot_2["metric_ci"])
pd.testing.assert_frame_equal(boot_1["delta_ci"], boot_2["delta_ci"])
boot_1["delta_ci"].head()

,mode,model,baseline,metric,delta_estimate,ci_low,ci_high,significant_95
12,macro,Residual XGB ETS LW,ETS local,HL-MAE,3.000000,2.258333,4.00,True
15,macro,Residual XGB ETS LW,ETS local,HL-asymMAE,4.666667,4.000000,6.00,True
13,macro,Residual XGB ETS LW,ETS local,HL-underMAE,1.666667,0.000000,3.00,False
14,macro,Residual XGB ETS LW,ETS local,HL-underRate,0.000000,0.000000,0.00,False
11,macro,Residual XGB ETS LW,ETS local,LW-MAE,2.683333,1.932500,3.75,True


## Optional full run

In [8]:
RUN_FULL_EXPERIMENT = True

if RUN_FULL_EXPERIMENT:
    DATA_ROOT = PROJECT_ROOT / "data"
    if not list(DATA_ROOT.glob("REF_PSG_PLACES*.csv")) and (PROJECT_ROOT.parent / "data").exists():
        DATA_ROOT = PROJECT_ROOT.parent / "data"

    full_cfg = MultiStationForecastConfig(
        data_dir=str(DATA_ROOT),
        cache_dir=str(DATA_ROOT / "cache"),
        output_dir=str(PROJECT_ROOT / "outputs" / "notebook_forecasting_run"),
        n_bootstrap=100,
        run_bootstrap=True,
    )

    result = run_multistation_forecasting_experiment(full_cfg)
    paths = save_multistation_forecasting_result(result)
    print(paths)
else:
    print("Fast repository check completed. Set RUN_FULL_EXPERIMENT = True for a real forecasting run.")

[timing] load_or_build_station_buckets: 0.19 s (0.00 min)
[timing] make_multistation_supervised_frame: 1.24 s (0.02 min)
[timing] make_multistation_weekday_split: 1.44 s (0.02 min)
[timing] prepare_feature_lists_and_seasonal_anchor: 0.00 s (0.00 min)
[timing] MegaBoost base: 38.47 s (0.64 min)
[timing] MegaBoost rich: 42.81 s (0.71 min)
[timing] MegaBoost rich LW: 41.46 s (0.69 min)
[timing] MegaBoost base norm: 36.23 s (0.60 min)
[timing] MegaBoost rich norm: 44.02 s (0.73 min)
[timing] MegaBoost rich norm LW: 42.92 s (0.72 min)
[timing] MegaBoost Seasonal residual: 34.07 s (0.57 min)
[timing] MegaBoost Seasonal residual LW: 40.94 s (0.68 min)
[timing] LineBoost rich norm LW: 13.22 s (0.22 min)
[timing] LineBoost Seasonal residual LW: 10.78 s (0.18 min)
[timing] StationBias Seasonal LW: 0.05 s (0.00 min)
[timing] CatBoost base: 20.84 s (0.35 min)
[timing] CatBoost rich: 26.00 s (0.43 min)
[timing] Km-XGBoost-lite: 24.74 s (0.41 min)
[timing] ET-LGBM-PSO-lite: 65.83 s (1.10 min)
[timin